# ZS601 3DGS v3 标准化预处理与实验
本 notebook 明确展示环境、输入路径、预处理、200 步冒烟、正式训练、最终评估和 Drive 回读。正式运行优先使用 NVIDIA L4；任何输出目录必须是新的。


## 1 环境 Python CUDA 与 L4 检查


In [ ]:
import os,sys,subprocess,json,platform
print("Python",sys.version)
print("Platform",platform.platform())
subprocess.run(["nvidia-smi"],check=True)
gpu=subprocess.check_output(["nvidia-smi","--query-gpu=name,memory.total,driver_version","--format=csv,noheader"],text=True).strip()
print("GPU",gpu)
if "L4" not in gpu:
    raise RuntimeError("正式流程要求 NVIDIA L4；请在 Colab 更改运行时后重试")


## 2 挂载 Drive 并配置不可变输入与新输出


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
SCENE_NAME="ZS601meetingroom"  # 按实际场景目录修改
DRIVE_SCENE=Path("/content/drive/MyDrive/LCCDataset/zs601_output")/SCENE_NAME
LOCAL_SCENE=Path("/content/datasets")/SCENE_NAME
DRIVE_RUN_ROOT=Path("/content/drive/MyDrive/LCCDataset/zs601_output")/"zs601_v3_standard_runs"
EXPERIMENT_GROUP="E"  # original A B C D E custom
RUN_ID="E_standard_150k_CHANGE_ME"
DRIVE_OUTPUT=DRIVE_RUN_ROOT/RUN_ID
LIDAR_RELATIVE=Path("raw/lidar/lidar_static_rgb.ply")  # 已去动态目标的彩色 PLY
if DRIVE_OUTPUT.exists():
    raise FileExistsError(f"输出目录已存在，必须使用新 RUN_ID: {DRIVE_OUTPUT}")
print(json.dumps({"drive_scene":str(DRIVE_SCENE),"local_scene":str(LOCAL_SCENE),
                  "group":EXPERIMENT_GROUP,"drive_output":str(DRIVE_OUTPUT)},indent=2))


## 3 获取代码 固定分支或提交


In [ ]:
REPO_URL="https://github.com/VISjudy/ZS601_3DGS.git"
CODE_REF="v3-experiment-standard"  # 正式发布后改为已验证的不可变 commit
REPO=Path("/content/ZS601_3DGS")
if REPO.exists():
    raise FileExistsError(f"拒绝覆盖已有代码目录: {REPO}")
subprocess.run(["git","clone","--branch",CODE_REF,"--single-branch",REPO_URL,str(REPO)],check=True)
CODE=REPO/"gaussian-splattingWithMask_v3"
head=subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip()
print("CODE",CODE,"HEAD",head)


## 4 安装依赖 构建 CUDA 扩展 并运行测试


In [ ]:
os.chdir(CODE)
subprocess.run([sys.executable,"-m","pip","install","-q","plyfile","laspy","scipy","pillow"],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","submodules/diff-gaussian-rasterization"],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","submodules/simple-knn"],check=True)
tests=["test_experiment_presets_v3","test_preprocess_v3","test_geometry_v3",
       "test_scale_bounds_v3","test_surface_densify_v3","test_lidar_depth_v3"]
subprocess.run([sys.executable,"-m","unittest","-v",*tests],check=True)


## 5 从 Drive 复制输入到 Colab 本地


In [ ]:
import shutil,time
if LOCAL_SCENE.exists():
    raise FileExistsError(f"拒绝覆盖本地输入: {LOCAL_SCENE}")
t=time.time()
shutil.copytree(DRIVE_SCENE,LOCAL_SCENE)
print("copied_seconds",time.time()-t)
print("files",sum(1 for p in LOCAL_SCENE.rglob("*") if p.is_file()))


## 6 SfM 数据校验与固化


In [ ]:
PROCESSED=LOCAL_SCENE/"processed_v3"
RAW_IMAGES=LOCAL_SCENE/"raw/images"
RAW_MASKS=LOCAL_SCENE/"raw/masks"
RAW_SPARSE=LOCAL_SCENE/"raw/camera_metadata/sparse/0"
LIDAR_PLY=LOCAL_SCENE/LIDAR_RELATIVE
common=[sys.executable,str(CODE/"scripts/preprocess_dataset_v3.py"),
        "--experiment-group",EXPERIMENT_GROUP,"--scene-root",str(LOCAL_SCENE),
        "--output",str(PROCESSED),"--images",str(RAW_IMAGES),"--masks",str(RAW_MASKS),
        "--sparse",str(RAW_SPARSE),"--lidar",str(LIDAR_PLY)]
MASK_VALID_WHEN="black"  # ZS601: black=valid, white=excluded; loader inverts it
subprocess.run(common+["--stage","sfm","--mask-valid-when",MASK_VALID_WHEN],check=True)


## 7 去动态目标 LiDAR 点云 法向估计与相机辅助定向


In [ ]:
subprocess.run(common+["--stage","lidar",
    "--estimate-normals","auto","--orient-normals-camera","auto",
    "--orientation-occlusion","auto","--propagate-unresolved","off",
    "--filter-outliers","off","--knn","24","--min-neighbors","8",
    "--max-curvature","0.15","--camera-count","8"],check=True)


## 8 生成深度图 法向图 并完成三维回投验证


In [ ]:
subprocess.run(common+["--stage","supervision",
    "--generate-supervision","auto","--near","0.1","--far","15.0",
    "--backprojection-samples","4096"],check=True)
subprocess.run(common+["--stage","validate"],check=True)
manifest=PROCESSED/"manifests/dataset_manifest.json"
print(manifest.read_text()[:12000])


## 9 将预处理结果保存回 Drive 新目录


In [ ]:
DRIVE_PROCESSED=DRIVE_SCENE/"processed_v3"
if DRIVE_PROCESSED.exists():
    raise FileExistsError(f"Drive 预处理目录已存在，先人工核对，禁止覆盖: {DRIVE_PROCESSED}")
shutil.copytree(PROCESSED,DRIVE_PROCESSED)
print("saved",DRIVE_PROCESSED)


## 10 公共训练参数 重点列出与原始 3DGS 默认不同项


In [ ]:
TRAIN_SOURCE=PROCESSED
POINT_CLOUD=PROCESSED/"geometry/lidar_static_rgb_normal_oriented.ply"
SPARSE=PROCESSED/"sparse/0"
DATA_MANIFEST=PROCESSED/"manifests/dataset_manifest.json"
COMMON_TRAIN=[
 sys.executable,str(CODE/"train_mask_v3.py"),
 "--experiment",EXPERIMENT_GROUP,
 "-s",str(TRAIN_SOURCE),"--point_cloud",str(POINT_CLOUD),
 "--train_file",str(SPARSE/"images-train.txt"),
 "--val_file",str(SPARSE/"images-val10.txt"),
 "--test_file",str(SPARSE/"images-test.txt"),
 "--cameras_file",str(SPARSE/"cameras.txt"),
 "--data_manifest",str(DATA_MANIFEST),"--supervision_root",str(PROCESSED),
 "--iterations","150000","--sh_degree","2",
 "--position_lr_init","0.000016","--position_lr_final","0.00000016",
 "--position_lr_max_steps","150000","--scaling_lr","0.0015",
 "--seed","42","--lazy_cache","100","--val_interval","5000",
 "--checkpoint_interval","50000","--val_npz","off",
 "--validation_diagnostics","on","--val_rgb","on","--val_depth","on",
 "--val_normal","on","--val_ellipsoids","on"]
if EXPERIMENT_GROUP=="E":
    COMMON_TRAIN += ["--lidar_depth_cache","/content/lidar_depth_cache",
                     "--lidar_depth_export",str(DRIVE_OUTPUT/"lidar_depth_pseudogt")]
print("与默认不同: iterations=150000 sh_degree=2 position_lr=1.6e-5->1.6e-7 "
      "lr_horizon=150000 scaling_lr=0.0015 val=5000 checkpoint=50000")
print("command"," ".join(COMMON_TRAIN))


## 11 200 步冒烟 使用正式 150k 学习率日程


In [ ]:
SMOKE_LOCAL=Path("/content/zs601_runs")/(RUN_ID+"_smoke200")
smoke=COMMON_TRAIN.copy()
smoke[smoke.index("--iterations")+1]="200"
smoke[smoke.index("--position_lr_max_steps")+1]="150000"
smoke += ["-m",str(SMOKE_LOCAL),"--final_test","off"]
subprocess.run(smoke,check=True)
subprocess.run([sys.executable,str(CODE/"scripts/verify_run_outputs_v3.py"),
                str(SMOKE_LOCAL),"--iterations","200","--val-interval","5000"],check=True)


## 12 正式 150000 步训练 输出先写 Colab 本地


In [ ]:
FORMAL_LOCAL=Path("/content/zs601_runs")/RUN_ID
formal=COMMON_TRAIN+["-m",str(FORMAL_LOCAL),"--final_test","on"]
subprocess.run(formal,check=True)


## 13 最终测试 31 个 val 时间点 3 个 checkpoint 与最差 10 相机验收


In [ ]:
subprocess.run([sys.executable,str(CODE/"scripts/verify_run_outputs_v3.py"),
                str(FORMAL_LOCAL)],check=True)
print((FORMAL_LOCAL/"completed.json").read_text())
print((FORMAL_LOCAL/"experiment_summary.md").read_text()[:12000])


## 14 复制正式结果回 Drive 并回读验证


In [ ]:
if DRIVE_OUTPUT.exists():
    raise FileExistsError(f"拒绝覆盖 Drive 输出: {DRIVE_OUTPUT}")
shutil.copytree(FORMAL_LOCAL,DRIVE_OUTPUT)
checks=["completed.json","run_config.json","loss_log.csv","training_progress.csv",
        "val_metrics.csv","geometry_metrics.csv","test_final/iteration_150000/test_metrics_per_camera.csv",
        "results_table.csv","results_table.md","results_table.tex","experiment_summary.md"]
missing=[x for x in checks if not (DRIVE_OUTPUT/x).is_file()]
print("missing",missing)
if missing: raise RuntimeError(missing)
subprocess.run([sys.executable,str(CODE/"scripts/verify_run_outputs_v3.py"),
                str(DRIVE_OUTPUT)],check=True)
print("DRIVE_VERIFIED",DRIVE_OUTPUT)


## 15 完成后手动断开运行时
只有上一单元回读验证成功后才在 Colab 菜单中断开并删除运行时；不要在复制或验证过程中断开 GPU。
